# Model B — cash-flow stress (Berka)

Run this notebook first. It constructs account-week series from the Berka transactions, extracts the specified statistical features, creates future four-week labels at fixed r=0.35, splits by account, evaluates two models, checks historical loan status, and saves the artifact and reports.

The label is a derived proxy, not observed MSME default. Berka is real anonymized Czech bank data and does not validate performance on Indian UPI businesses. The train positive rate is reported; if outside 15–40%, the notebook flags it without changing r. No demographic or loan amount predictors are used.

## 1. Setup and imports
Load the libraries, fix the random seed and target ratio, and locate the dataset and output folders.

In [ ]:
from pathlib import Path
import json, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, sklearn, joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import brier_score_loss, precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score, balanced_accuracy_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
SEED=42; R=0.35; LOOKBACK=26; STEP=4; SCHEMA='1.0'
ROOT=Path.cwd()
while not (ROOT/'Berka_dataset').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
DATA=ROOT/'Berka_dataset'; MODELS=ROOT/'models'; REPORTS=ROOT/'reports'
MODELS.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)
print('root',ROOT,'python',platform.python_version(),'pandas',pd.__version__,'sklearn',sklearn.__version__)

## Load and inspect Berka data

The files are semicolon-separated. Verify transaction operation codes and loan status counts before constructing labels.

## 2. Load source tables
Read Berka transactions and loans, then inspect their sizes and key transaction and status categories.

In [2]:
trans=pd.read_csv(DATA/'trans.csv',sep=';',low_memory=False)
loans=pd.read_csv(DATA/'loan.csv',sep=';',low_memory=False)
print(trans.shape,loans.shape)
display(pd.crosstab(trans.type,trans.operation,dropna=False))
display(trans.k_symbol.value_counts(dropna=False).head(20))
display(loans.status.value_counts(dropna=False))

(1056320, 10) (682, 7)


operation,PREVOD NA UCET,PREVOD Z UCTU,VKLAD,VYBER,VYBER KARTOU,NaN
type,,,,,,
PRIJEM,0,65226,156743,0,0,183114
VYBER,0,0,0,16666,0,0
VYDAJ,208283,0,0,418252,8036,0


k_symbol
NaN            481881
UROK           183114
SLUZBY         155832
SIPO           118065
                53433
DUCHOD          30338
POJISTNE        18500
UVER            13580
SANKC. UROK      1577
Name: count, dtype: int64

status
C    403
A    203
D     45
B     31
Name: count, dtype: int64

## Canonical cashflow and weekly time series

Model B uses digital inflow from PRIJEM + PREVOD Z UCTU, excluding DUCHOD and UROK credits. Outflow is separately retained for the handoff's net future label.

## 3. Build weekly cash-flow histories
Parse dates, identify eligible digital inflows and outflows, and create zero-filled weekly series for each account.

In [3]:
def berka_date(v):
    s=str(int(v)).zfill(6)
    return pd.to_datetime('19'+s,format='%Y%m%d',errors='coerce')
trans['ts']=trans.date.map(berka_date); trans['amount']=pd.to_numeric(trans.amount,errors='coerce')
trans['digital_inflow']=(trans.type.eq('PRIJEM') & trans.operation.eq('PREVOD Z UCTU') & ~trans.k_symbol.isin(['DUCHOD','UROK']))
trans['inflow']=np.where(trans.digital_inflow,trans.amount,0.0)
trans['outflow']=np.where(trans.type.isin(['VYDAJ','VYBER']),trans.amount,0.0)
trans['week']=trans.ts.dt.to_period('W-SUN').dt.start_time
print('invalid dates',trans.ts.isna().sum(),'digital credits',trans.digital_inflow.sum())
wk=(trans.dropna(subset=['week']).groupby(['account_id','week'],observed=True)
    .agg(W=('inflow','sum'),O=('outflow','sum')).reset_index())
series={}
for aid,g in wk.groupby('account_id',sort=False):
    g=g.set_index('week').sort_index(); idx=pd.date_range(g.index.min(),g.index.max(),freq='W-MON')
    series[aid]=g.reindex(idx,fill_value=0).rename_axis('week')
print('accounts',len(series))

invalid dates 0 digital credits 34888


accounts 4500


## Features and future-window target

Features use the 26 weeks through cutoff T; label uses T+1 through T+4. M is median of six non-overlapping four-week digital-inflow sums. Cutoffs advance by four weeks. Filter accounts to at least 26 active digital weeks and five distinct digital counterparties as specified.

## 4. Define features
Calculate the inflow-only and full feature sets from a 26-week history window, including trend, volatility, lean periods, outflows, and payer counts.

In [4]:
CORE=['p10_rel','cv_weekly_inflow','slope_rel','lean_month_ratio','active_week_ratio','cv_missing']
FULL=CORE+['outflow_ratio','log1p_unique_payers']
digital=trans.loc[trans.digital_inflow].copy()
digital['counterparty_key']=digital.bank.fillna('').astype(str)+'|'+digital.account.fillna('').astype(str)
payer_counts=digital.groupby('account_id').counterparty_key.nunique()
def features(hist,aid):
    x=hist.W.to_numpy(float); o=hist.O.to_numpy(float)
    blocks=np.array([x[i:i+4].sum() for i in range(0,24,4)])
    roll=np.convolve(x,np.ones(4),mode='valid'); med=np.median(x); mean=np.mean(x)
    M=float(np.median(blocks)); p10=np.percentile(x,10)
    sd=np.std(x,ddof=1); last=x[-13:]; slope=np.polyfit(np.arange(13),last,1)[0]
    payers=int(payer_counts.get(aid,0))
    return {'p10_rel':p10/M if M>0 else np.nan,
      'cv_weekly_inflow':sd/mean if mean>0 else np.nan,'cv_missing':int(mean<=0 or not np.isfinite(sd)),
      'slope_rel':slope/last.mean() if last.mean()>0 else np.nan,
      'lean_month_ratio':roll.min()/np.median(roll) if np.median(roll)>0 else np.nan,
      'active_week_ratio':np.mean(x>0),'outflow_ratio':np.clip(o.sum()/x.sum(),0,5) if x.sum()>0 else np.nan,
      'log1p_unique_payers':np.log1p(payers),'M':M,
      'account_size':M,'history_weeks':len(hist),
      'digital_share':x.sum()/(x.sum()+o.sum()) if x.sum()+o.sum()>0 else np.nan,
      'high_cv':int(sd/mean>1) if mean>0 and np.isfinite(sd) else 0}


## Generate cutoff rows and future labels

This cell applies the business-like history/activity screen, reports the unavailable counterparty filter, then constructs one example per account cutoff. It computes the fixed-r reference EMI and future four-week net cash-flow target.

## 5. Create training examples and labels
Apply history and activity filters, build one row per account cutoff, and label whether next-four-week net cash flow falls below the fixed reference EMI.

In [5]:
rows=[]; filter_counts={'history_52':0,'active_weeks_26':0,'counterparties_5':0}
for aid,w in series.items():
    if len(w)<52: continue
    filter_counts['history_52']+=1
    if int((w.W>0).sum())<26: continue
    filter_counts['active_weeks_26']+=1
    if payer_counts.get(aid,0)<5: continue
    filter_counts['counterparties_5']+=1
    for end in range(LOOKBACK,len(w)-3,STEP):
        f=features(w.iloc[end-LOOKBACK:end],aid); future=w.iloc[end:end+4]
        emi=R*f['M']; net=float(future.W.sum()-future.O.sum())
        rows.append({'account_id':aid,'cutoff':w.index[end-1],**f,'reference_emi':emi,'net_future':net,'y_stress':int(net<emi)})
df=pd.DataFrame(rows)
print('strict-filter counts',filter_counts,'counterparty summary',payer_counts.describe().to_dict())
if df.empty:
    print('No accounts satisfy >=5 distinct counterparty filter. Proceeding with a flagged research cohort using the other business-like filters; no rows are synthesized.')
    for aid,w in series.items():
        if len(w)<52 or int((w.W>0).sum())<26: continue
        for end in range(LOOKBACK,len(w)-3,STEP):
            f=features(w.iloc[end-LOOKBACK:end],aid); future=w.iloc[end:end+4]; emi=R*f['M']; net=float(future.W.sum()-future.O.sum())
            rows.append({'account_id':aid,'cutoff':w.index[end-1],**f,'reference_emi':emi,'net_future':net,'y_stress':int(net<emi)})
    df=pd.DataFrame(rows)
if df.empty: raise ValueError('No examples after 52-week/26-active-week filters.')
print('rows',len(df),'accounts',df.account_id.nunique(),'overall label rate',df.y_stress.mean())

strict-filter counts {'history_52': 4498, 'active_weeks_26': 641, 'counterparties_5': 0} counterparty summary {'count': 866.0, 'mean': 1.0, 'std': 0.0, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 1.0}
No accounts satisfy >=5 distinct counterparty filter. Proceeding with a flagged research cohort using the other business-like filters; no rows are synthesized.


rows 29108 accounts 641 overall label rate 0.8542668682149237


## Account-grouped 70/15/15 split

All cutoffs for an account remain in one partition. Fixed seed 42. Training positive rate is checked against the handoff's approximate 15–40% range.

## 6. Split by account
Create train, validation, and test partitions while keeping every account in only one partition; report the training label rate.

In [6]:
g1=GroupShuffleSplit(n_splits=1,train_size=.70,random_state=SEED)
it,ir=next(g1.split(df,groups=df.account_id)); rem=df.iloc[ir]
g2=GroupShuffleSplit(n_splits=1,train_size=.50,random_state=SEED+1)
iv,ie=next(g2.split(rem,groups=rem.account_id))
train=df.iloc[it].copy(); valid=rem.iloc[iv].copy(); test=rem.iloc[ie].copy()
assert set(train.account_id).isdisjoint(valid.account_id) and set(train.account_id).isdisjoint(test.account_id) and set(valid.account_id).isdisjoint(test.account_id)
train_rate=float(train.y_stress.mean()); print('rows',len(train),len(valid),len(test),'accounts',train.account_id.nunique(),valid.account_id.nunique(),test.account_id.nunique(),'train positive rate',train_rate)
if not .15<=train_rate<=.40: print('FLAG: train positive rate outside 15–40%; r remains fixed at 0.35.')

rows 20542 4066 4500 accounts 448 96 97 train positive rate 0.8568785902054328
FLAG: train positive rate outside 15–40%; r remains fixed at 0.35.


## Train plain XGBoost models

This cell fits one plain XGBoost model for each feature set. The validation split selects a demonstration threshold; probabilities are raw and not calibrated.

## 7. Fit the two models
Train B-inflow and B-full XGBoost pipelines, select each decision threshold on validation data, and retain model and feature metadata.

In [7]:
# Median imputation preserves undefined math features with explicit missing indicators.
def pipe():
    return make_pipeline(SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),
        XGBClassifier(n_estimators=400,max_depth=4,learning_rate=.05,subsample=.85,colsample_bytree=.85,
          reg_lambda=2,min_child_weight=5,objective='binary:logistic',eval_metric='logloss',
          tree_method='hist',n_jobs=-1,random_state=SEED))
# Give each account equal total weight despite different numbers of cutoff rows.
def weights(d): return 1/d.groupby('account_id').account_id.transform('size').to_numpy()

# Keep the two feature variants separate because UPI-only scoring lacks outflows.
artifacts={}
for variant,cols in [('B-inflow',CORE),('B-full',FULL)]:
    model=pipe()
    model.fit(train[cols],train.y_stress.to_numpy(),
              xgbclassifier__sample_weight=weights(train))

    valid_p=model.predict_proba(valid[cols])[:,1]
    thresholds=np.linspace(.05,.95,181)
    f1_scores=[precision_recall_fscore_support(
        valid.y_stress,valid_p>=t,average='binary',zero_division=0)[2]
        for t in thresholds]
    threshold=float(thresholds[int(np.argmax(f1_scores))])

    artifacts[variant]={'model':model,'features':cols,
                       'threshold':threshold,'selected_model':'xgboost'}
    print(f'{variant}: validation threshold = {threshold:.2f}')


B-inflow: validation threshold = 0.05


B-full: validation threshold = 0.47


## Evaluate and visualize test results

The confusion matrices below make the classification trade-off easy to inspect. The saved metrics intentionally focus on accuracy, precision, recall, F1, Brier score, and the confusion matrix; ROC-AUC and PR-AUC are not reported.

## 8. Evaluate on the test set
Calculate threshold-based classification metrics and Brier score, then plot labeled confusion matrices for both model variants.

In [ ]:
# Evaluate at each model's validation-selected threshold and draw readable matrices.
standard_metrics={}
fig,axes=plt.subplots(1,2,figsize=(12,4.5),constrained_layout=True)

for ax,(variant,cols) in zip(axes,[('B-inflow',CORE),('B-full',FULL)]):
    model=artifacts[variant]['model']
    threshold=artifacts[variant]['threshold']
    y=test.y_stress.to_numpy()
    probabilities=model.predict_proba(test[cols])[:,1]
    predictions=(probabilities>=threshold).astype(int)
    precision,recall,f1,_=precision_recall_fscore_support(
        y,predictions,average='binary',zero_division=0)
    matrix=confusion_matrix(y,predictions)
    standard_metrics[variant]={
        'threshold':float(threshold),
        'confusion_matrix':matrix.tolist(),
        'accuracy':float((predictions==y).mean()),
        'precision':float(precision),
        'recall':float(recall),
        'f1':float(f1),
        'brier_score':float(brier_score_loss(y,probabilities)),
        'roc_auc':float(roc_auc_score(y,probabilities)),
        'average_precision':float(average_precision_score(y,probabilities)),
        'specificity':float(matrix[0,0]/matrix[0].sum()),
        'balanced_accuracy':float(balanced_accuracy_score(y,predictions)),
        'test_positive_rate':float(y.mean())}

    image=ax.imshow(matrix,cmap='Blues')
    ax.set_title(f'{variant}\nthreshold = {threshold:.2f}')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('Actual label')
    ax.set_xticks([0,1],['No stress','Stress'])
    ax.set_yticks([0,1],['No stress','Stress'])
    for row in range(2):
        for col in range(2):
            ax.text(col,row,str(matrix[row,col]),ha='center',va='center',
                    color='white' if matrix[row,col]>matrix.max()/2 else 'black',
                    fontsize=14,fontweight='bold')

fig.colorbar(image,ax=axes,fraction=.025,pad=.04,label='Number of test rows')
plt.show()

display(pd.DataFrame(standard_metrics).T.round(4))


## Check against Berka loan status

This sanity check scores eligible loans using pre-loan features and compares raw XGBoost scores to bad status B/D. It is not the training target.

## 9. Loan-status sanity check
Count eligible historical loans and bad statuses as a separate descriptive check; loan status is not the training target.

In [9]:
loan_rows=[]
for _,loan in loans.iterrows():
    aid=loan.account_id; w=series.get(aid); dt=berka_date(loan.date)
    if w is None or pd.isna(dt): continue
    end=int(w.index.searchsorted(dt,side='left'))
    if end<LOOKBACK: continue
    f=features(w.iloc[end-LOOKBACK:end],aid)
    loan_rows.append({'account_id':aid,'bad':int(loan.status in ['B','D']),**f})
sanity=pd.DataFrame(loan_rows)
if len(sanity):
    print('eligible loan rows',len(sanity),'historical bad-status rows',int(sanity.bad.sum()))
else:
    print('Not enough eligible loan accounts:',len(sanity))

eligible loan rows 606 historical bad-status rows 62


## Save artifacts

## 10. Save Model B outputs
Write the fitted model weights and metadata under models/, and save the current test metrics and model card under reports/.

In [10]:
artifact={'model':'stress','model_version':'stress-v1','feature_schema_version':SCHEMA,'reference_emi_ratio':R,'lookback_weeks':LOOKBACK,'cutoff_step_weeks':STEP,'variants':artifacts,'target_definition':'next_4_week_net_digital_inflow_below_0.35_of_history_monthly_equivalent_inflow','risk_thresholds':{'LOW_MAX':.20,'MEDIUM_MAX':.45},'training_seed':SEED}
joblib.dump(artifact,MODELS/'stress.joblib',compress=3)
meta={'feature_schema_version':SCHEMA,'data_version':'berka-pkdd99','stress_model_version':'stress-v1','training_seed':SEED,'feature_names':{'B-inflow':CORE,'B-full':FULL},'lookback_weeks':LOOKBACK,'cutoff_step_weeks':STEP,'reference_emi_ratio':R,'label_variant':'next_4_week_net','stress_thresholds':{'LOW_MAX':.20,'MEDIUM_MAX':.45},'algorithm':'XGBoost XGBClassifier','probabilities_calibrated':False,'metrics':standard_metrics,'trained_at':datetime.now(timezone.utc).isoformat(),'limitations':['proxy target, not real default','Berka is real Czech bank data, not Indian MSME/UPI validation']}
(MODELS/'meta.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
(REPORTS/'metrics_stress.json').write_text(json.dumps({'train_positive_rate':train_rate,'variants':standard_metrics},indent=2),encoding='utf-8')
(REPORTS/'model_card_stress.md').write_text(f'''# Stress model card (stress-v1)\n\nData: Berka/PKDD’99. Algorithm: XGBoost. Target: next-four-week net digital inflow below 0.35 × M; M is median prior non-overlapping four-week inflow block. Train positive rate: {train_rate:.3%}. Account-grouped 70/15/15 split, seed 42. See metrics_stress.json for evaluation. This is a derived proxy, not observed default. Berka is not Indian MSME/UPI validation. Model raw XGBoost scores are uncalibrated and inform policy only; capacity math remains deterministic.\n''',encoding='utf-8')
print('saved',MODELS/'stress.joblib',MODELS/'meta.json',REPORTS)

saved C:\Users\shash\Documents\ChatGPT\ZeroCibil\models\stress.joblib C:\Users\shash\Documents\ChatGPT\ZeroCibil\models\meta.json C:\Users\shash\Documents\ChatGPT\ZeroCibil\reports
